# Final Project - Wind Turbine Fault Detection

This notebook presents the complete, end-to-end workflow for a wind turbine fault detection project using multivariate SCADA time-series data. The goal is to predict whether a turbine is approaching a failure event within a future prediction horizon of 24, 48, or 72 hours.

The final report explains the motivation, methodology, and interpretation of the results in narrative form. This notebook complements that report by showing how the analysis was implemented: data loading, preprocessing, feature harmonization, label construction, modeling, evaluation, visualization, and report-output generation.

Although the project codebase is organized as a modular Python package, this notebook consolidates the full pipeline into a single reproducible submission artifact. Artifact checks are used throughout so that expensive preprocessing and modeling steps can be reused when valid outputs already exist.


## How this notebook is organized

The notebook follows the same analytical sequence as the final report:

1. Configure the local project environment and reusable notebook controls.
2. Validate the expected repository layout and input data locations.
3. Run or reuse preprocessing artifacts that convert raw SCADA/event files into a consolidated modeling dataset.
4. Inspect the processed data and document key preprocessing choices.
5. Generate report-ready figures that explain feature harmonization, labeling, class imbalance, model performance, threshold behavior, temporal prediction behavior, and feature importance.
6. Run or reuse modeling experiments for the 24h, 48h, and 72h prediction horizons.
7. Summarize results and save output files needed for the final report.

This structure is intentionally more linear than a production research workflow. In practice, EDA, preprocessing, feature engineering, modeling, and evaluation would often remain in separate notebooks or scripts to avoid unnecessary recomputation. For the final course submission, however, the full workflow is shown in one notebook while still relying on the modular `wtfd` package for maintainability.


In [1]:
# Optional notebook convenience magics.
# These can be commented out if your environment does not support IPython magics.
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import gc
import json
import sys
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import Markdown, display
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import PercentFormatter

In [3]:
def find_project_root(start: Optional[Path] = None) -> Path:
    """Walk upward until a repository root is found."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from within the ML_Project repository."
    )

PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source root:  {SRC_ROOT}")


Project root: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project
Source root:  /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/src


In [4]:
# Optional editable install for environments where the package has not been installed yet.
# Set to True only if imports fail below and you want the notebook to install the local package.
ENABLE_EDITABLE_INSTALL_IF_NEEDED = False

if ENABLE_EDITABLE_INSTALL_IF_NEEDED:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
        check=True,
    )


In [5]:
from wtfd.data.preprocessing import WindFarmProcessor
from wtfd.models.artifacts import (
    ensure_run_output_dir,
    load_dataframe_artifact,
    load_json_artifact,
    save_dataframe_artifact,
    save_feature_importance,
    save_model_metrics,
    save_run_metadata,
    save_threshold_sweep,
)
from wtfd.models.experiments import get_experiment_config, list_available_experiments
from wtfd.models.feature_selector import (
    build_feature_matrix,
    summarize_feature_selection,
    validate_no_leakage_columns_in_features,
)
from wtfd.models.metrics import build_threshold_sweep_table
from wtfd.models.model_registry import get_model_config, list_available_models
from wtfd.models.splitter import WindFarmSplitter
from wtfd.models.trainer import WindFaultTrainer
from wtfd.utils.logging_utils import get_logger

logger = get_logger("wtfd.notebooks.final_project")

try:
    import xgboost  # noqa: F401
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Reset plotting defaults before applying the project-wide style.
# This prevents prior notebook runs or imported libraries from leaking grid/style settings.
plt.rcParams.update(plt.rcParamsDefault)
sns.set_theme(style="white", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

print("Available experiments:", list_available_experiments())
print("Available models:", list_available_models())
print("XGBoost available:", HAS_XGBOOST)


Available experiments: ['pre_24h', 'pre_48h', 'pre_72h']
Available models: ['logistic', 'rf', 'xgboost']
XGBoost available: True


## Notebook controls

The following configuration cell centralizes paths, runtime toggles, and report settings. The defaults favor reproducibility and efficiency:

- existing preprocessing and modeling artifacts are reused when available;
- expensive steps can be forced to rebuild by changing the relevant flags;
- report figures are written to `outputs/final_report_figures` using stable filenames.

The buffer settings are part of the event-based labeling design. A pre-event buffer is applied before the earliest prediction window, and a post-event buffer is applied after the event window to reduce label ambiguity and potential leakage.


In [6]:
# ------------------------------------------------------------------
# Project paths
# These paths match the repository structure used by the package and report.
# ------------------------------------------------------------------
CONFIG_PATH = PROJECT_ROOT / "config" / "feature_map.yaml"
RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "zenodo_windfarm_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MASTER_DATASET_PATH = PROCESSED_DIR / "master_dataset.parquet"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "modeling"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
TIMELINE_DIR = OUTPUT_DIR / "failure_timelines"
REPORT_FIG_DIR = OUTPUT_DIR / "final_report_figures"
REPORT_DIAGNOSTICS_DIR = PROJECT_ROOT / "artifacts" / "report_diagnostics"

# ------------------------------------------------------------------
# Preprocessing controls
# Rebuild flags are intentionally False by default because preprocessing is expensive.
# ------------------------------------------------------------------
# NOTE:
# The WTDF preprocessing code supports a pre-window exclusion buffer
# immediately before the earliest 72h positive window and a post-event
# exclusion buffer immediately after the event window ends.
BUFFER_BEFORE_HOURS = 4.0
BUFFER_AFTER_HOURS = 4.0
REBUILD_EVENT_PARQUETS = False
REBUILD_MASTER_DATASET = False

# ------------------------------------------------------------------
# Data inspection controls
# Sampling keeps early inspection fast; full data is loaded later only when needed.
# ------------------------------------------------------------------
SAMPLE_N_ROWS = 5000
HARMONIZATION_FEATURE_COUNT = 13
FORCE_REBUILD_HARMONIZATION_CACHE = False
HARMONIZATION_CACHE_PATH = REPORT_DIAGNOSTICS_DIR / "feature_harmonization_overview.csv"

# ------------------------------------------------------------------
# Modeling controls
# ------------------------------------------------------------------
EXPERIMENT_NAMES = ["pre_24h", "pre_48h", "pre_72h"]
MODEL_NAMES_OVERRIDE = None     # e.g. ["logistic", "rf", "xgboost"]
NUMERIC_ONLY = True
FEATURE_SUBSET = None
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42
REUSE_SAVED_MODEL_ARTIFACTS = True
SAVE_FRESH_MODELING_ARTIFACTS = False

# ------------------------------------------------------------------
# Report-specific controls
# ------------------------------------------------------------------
REPORT_EXPERIMENT_NAME = "pre_72h"
REPORT_MODEL_NAME = "xgboost"
FORCE_REBUILD_REPORT_DIAGNOSTICS = False
SAVE_REPORT_FIGURES = True
TABLE_DECIMALS = 3

# ------------------------------------------------------------------
# Visualization controls
# ------------------------------------------------------------------
TIMELINE_SIGNALS = [
    "vibration_raw",
    "nacelle_temp",
    "temp_delta_gearbox",
]
TIMELINE_HOURS_BEFORE = 168
TIMELINE_HOURS_AFTER = 24
TOP_N_FEATURES = 15

# Conceptual display only for Figure 1. This is not a data-derived duration.
FIG_1_EVENT_WINDOW_HOURS = 4.0

## Validate required repository inputs

Before running the pipeline, the notebook checks for the required project directories and configuration files. This provides an early failure point if the notebook is run outside the expected repository structure or before the raw data/configuration files have been placed correctly.


In [7]:
required_paths = {
    "config": CONFIG_PATH,
    "raw_data_root": RAW_DATA_ROOT,
    "processed_dir": PROCESSED_DIR,
    "artifact_root": ARTIFACT_ROOT,
    "output_dir": OUTPUT_DIR,
    "report_fig_dir": REPORT_FIG_DIR,
    "report_diagnostics_dir": REPORT_DIAGNOSTICS_DIR,
}

for path_name, path_value in required_paths.items():
    print(f"{path_name:>22}: {path_value} | exists={path_value.exists()}")

assert CONFIG_PATH.exists(), f"Missing config file: {CONFIG_PATH}"
assert RAW_DATA_ROOT.exists(), f"Missing raw data root: {RAW_DATA_ROOT}"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMELINE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

event_parquet_files = sorted(
    [p for p in PROCESSED_DIR.glob("*.parquet") if p.name != "master_dataset.parquet"]
)


def processed_dataset_has_excluded_buffer_rows(parquet_files: list[Path]) -> bool:
    """Return True if any processed event parquet contains excluded-buffer rows."""
    for parquet_path in parquet_files:
        try:
            buffer_df = pd.read_parquet(parquet_path, columns=["is_excluded_buffer"])
            if "is_excluded_buffer" in buffer_df.columns and buffer_df["is_excluded_buffer"].fillna(False).any():
                return True
        except Exception:
            try:
                state_df = pd.read_parquet(parquet_path, columns=["state_name"])
                if "state_name" in state_df.columns and state_df["state_name"].astype(str).eq("excluded_buffer").any():
                    return True
            except Exception:
                continue
    return False


existing_buffer_rows_present = (
    processed_dataset_has_excluded_buffer_rows(event_parquet_files)
    if event_parquet_files
    else False
)

repo_summary_df = pd.DataFrame(
    [
        {"item": "event parquet files", "count": len(event_parquet_files)},
        {"item": "master dataset exists", "count": int(MASTER_DATASET_PATH.exists())},
        {"item": "artifact experiment folders", "count": sum(1 for p in ARTIFACT_ROOT.glob("*") if p.is_dir())},
        {"item": "report figure directory", "count": int(REPORT_FIG_DIR.exists())},
        {"item": "existing excluded_buffer rows detected", "count": int(existing_buffer_rows_present)},
    ]
)
display(repo_summary_df)

if event_parquet_files and (BUFFER_BEFORE_HOURS > 0 or BUFFER_AFTER_HOURS > 0) and not existing_buffer_rows_present:
    print(
        "Configured notebook uses nonzero exclusion buffers, but the existing processed "
        "event parquet files contain no excluded_buffer rows. Forcing a preprocessing "
        "rebuild so the notebook, figure logic, and underlying labels stay aligned."
    )
    REBUILD_EVENT_PARQUETS = True
    REBUILD_MASTER_DATASET = True
    REUSE_SAVED_MODEL_ARTIFACTS = False
    FORCE_REBUILD_REPORT_DIAGNOSTICS = True

print()
print(f"BUFFER_BEFORE_HOURS = {BUFFER_BEFORE_HOURS}")
print(f"BUFFER_AFTER_HOURS  = {BUFFER_AFTER_HOURS}")
print(f"REBUILD_EVENT_PARQUETS = {REBUILD_EVENT_PARQUETS}")
print(f"REBUILD_MASTER_DATASET = {REBUILD_MASTER_DATASET}")
print(f"REUSE_SAVED_MODEL_ARTIFACTS = {REUSE_SAVED_MODEL_ARTIFACTS}")
print(f"FORCE_REBUILD_REPORT_DIAGNOSTICS = {FORCE_REBUILD_REPORT_DIAGNOSTICS}")

                config: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/config/feature_map.yaml | exists=True
         raw_data_root: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/raw/zenodo_windfarm_data | exists=True
         processed_dir: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed | exists=True
         artifact_root: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/artifacts/modeling | exists=True
            output_dir: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/outputs | exists=True
        report_fig_dir: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/outputs/final_report_figures | exists=True
report_diagnostics_dir: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/artifacts/report_diagnostics | exists=True


,item,count
0,event parquet files,95
1,master dataset exists,1
2,artifact experiment folders,3
3,report figure directory,1
4,existing excluded_buffer rows detected,1



BUFFER_BEFORE_HOURS = 4.0
BUFFER_AFTER_HOURS  = 4.0
REBUILD_EVENT_PARQUETS = False
REBUILD_MASTER_DATASET = False
REUSE_SAVED_MODEL_ARTIFACTS = True
FORCE_REBUILD_REPORT_DIAGNOSTICS = False


## Methodology summary

### Data processing strategy

The raw dataset contains SCADA measurements and event metadata from multiple wind farms. Because the raw files are large and farm schemas differ, preprocessing is performed at the turbine-event level before being consolidated into a master Parquet dataset. This design preserves temporal ordering, limits memory pressure, and keeps event metadata aligned with sensor measurements.

### Feature harmonization and engineering

The wind farms do not use a single shared sensor schema. The same physical concept may appear under different raw column names across farms, and some variables are unavailable for some farms. The preprocessing pipeline maps raw farm-specific channels into a standardized feature space, then creates derived temporal features such as temperature deltas, volatility measures, and efficiency proxies. These features are intended to capture gradual degradation rather than isolated sensor spikes.

### Labeling strategy

The prediction task is framed as binary classification. For each prediction horizon, an observation is labeled positive when a failure event occurs within the corresponding future window. Separate experiments are run for 24h, 48h, and 72h horizons. Buffer zones exclude ambiguous observations near event windows so that positive and negative samples better reflect the intended operational question.

### Modeling and evaluation strategy

Three supervised models are evaluated: Logistic Regression, Random Forest, and XGBoost. Logistic Regression provides an interpretable baseline, while Random Forest and XGBoost capture nonlinear relationships and feature interactions. Because failure observations are rare, model performance is evaluated with precision, recall, and F1 rather than accuracy. Classification thresholds are tuned on validation data to reflect the operational tradeoff between missed failures and false alarms.


## Initialize the preprocessing pipeline

The `WindFarmProcessor` loads the feature mapping configuration and applies the project’s preprocessing rules. The processor is responsible for harmonizing raw sensor names, assigning event-based state labels, applying buffer logic, creating derived features, and writing reusable Parquet outputs.


In [8]:
processor = WindFarmProcessor(
    config_path=CONFIG_PATH,
    buffer_before_hours=BUFFER_BEFORE_HOURS,
    buffer_after_hours=BUFFER_AFTER_HOURS,
)

processor


2026-04-15 11:34:13 | INFO | wtfd.data.preprocessing | Initialized WindFarmProcessor with config_path=/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/config/feature_map.yaml, buffer_before_hours=4.0, buffer_after_hours=4.0


## Run preprocessing only when needed

This cell performs the main data-processing step. If event-level Parquet files and the master dataset already exist, they are reused to avoid unnecessary reprocessing. If they are missing, or if the rebuild flags are enabled, the pipeline regenerates them from the raw SCADA and event files.

This approach keeps the notebook reproducible while acknowledging that full preprocessing can be time-consuming on the complete dataset.


In [9]:
if REBUILD_EVENT_PARQUETS or not event_parquet_files:
    logger.info("Running event-level preprocessing.")
    processor.process_all_turbines(
        raw_data_root=RAW_DATA_ROOT,
        output_dir=PROCESSED_DIR,
    )
    event_parquet_files = sorted(
        [p for p in PROCESSED_DIR.glob("*.parquet") if p.name != "master_dataset.parquet"]
    )
else:
    logger.info("Reusing existing event-level parquet files.")
    print(f"Reusing {len(event_parquet_files)} event-level parquet files from {PROCESSED_DIR}")

if REBUILD_MASTER_DATASET or not MASTER_DATASET_PATH.exists():
    if not event_parquet_files:
        raise FileNotFoundError(
            "No event-level parquet files are available, so the master dataset cannot be built."
        )
    logger.info("Building consolidated master dataset.")
    master_dataset_output = processor.create_master_dataset(
        processed_dir=PROCESSED_DIR,
        output_path=MASTER_DATASET_PATH,
    )
    print(f"Master dataset created: {master_dataset_output}")
else:
    logger.info("Reusing existing master dataset.")
    print(f"Reusing existing master dataset: {MASTER_DATASET_PATH}")

assert MASTER_DATASET_PATH.exists(), f"Master dataset not found: {MASTER_DATASET_PATH}"


2026-04-15 11:34:13 | INFO | wtfd.notebooks.final_project | Reusing existing event-level parquet files.
2026-04-15 11:34:13 | INFO | wtfd.notebooks.final_project | Reusing existing master dataset.


Reusing 95 event-level parquet files from /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed
Reusing existing master dataset: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet


## Review generated preprocessing artifacts

After preprocessing, each event-level Parquet file represents a processed turbine-event slice. The master dataset combines these processed files into a single modeling table. This summary confirms that the expected artifacts were produced and gives a quick view of the event-level output structure.


In [10]:
event_file_summary = pd.DataFrame(
    {
        "file_name": [p.name for p in event_parquet_files],
        "size_mb": [round(p.stat().st_size / (1024 ** 2), 3) for p in event_parquet_files],
    }
).sort_values("file_name").reset_index(drop=True)

display(event_file_summary.head(10))

master_parquet = pq.ParquetFile(MASTER_DATASET_PATH)
print(f"Master dataset row groups: {master_parquet.num_row_groups}")
print("Master dataset schema:")
print(master_parquet.schema)


,file_name,size_mb
0,A_event_0.parquet,7.362
1,A_event_10.parquet,7.208
2,A_event_13.parquet,7.201
3,A_event_14.parquet,7.330
4,A_event_17.parquet,7.416
5,A_event_22.parquet,7.061
6,A_event_24.parquet,7.407
7,A_event_25.parquet,7.373
8,A_event_26.parquet,7.203
9,A_event_3.parquet,7.455


Master dataset row groups: 96
Master dataset schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 time_stamp (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 farm_id (String);
  optional binary field_id=-1 asset_id (String);
  optional double field_id=-1 amb_temp;
  optional double field_id=-1 wind_speed;
  optional double field_id=-1 pitch_angle;
  optional double field_id=-1 active_power;
  optional double field_id=-1 gen_speed;
  optional double field_id=-1 gearbox_oil_temp;
  optional double field_id=-1 transformer_temp;
  optional double field_id=-1 nacelle_temp;
  optional double field_id=-1 hub_temp;
  optional double field_id=-1 yaw_error;
  optional double field_id=-1 vibration_raw;
  optional double field_id=-1 hydraulic_temp;
  optional double field_id=-1 generator_temp;
  optional double field_id=-1 temp_delta_gearbox;
  optional double field_id=-1 temp_t

## Load a small sample for exploratory inspection

A small sample from the master dataset is loaded first for quick inspection. This avoids loading the full processed dataset before basic schema checks are complete and makes it easier to verify that the notebook is connected to the expected data artifact.


In [11]:
sample_batches = []
rows_loaded = 0

for batch in master_parquet.iter_batches(batch_size=SAMPLE_N_ROWS):
    sample_batches.append(batch)
    rows_loaded += batch.num_rows
    if rows_loaded >= SAMPLE_N_ROWS:
        break

sample_df = pa.Table.from_batches(sample_batches).slice(0, SAMPLE_N_ROWS).to_pandas()

print(f"Loaded {len(sample_df):,} sample rows from {MASTER_DATASET_PATH.name}")
display(sample_df.head())
print()
sample_df.info()


Loaded 5,000 sample rows from master_dataset.parquet


,time_stamp,farm_id,asset_id,amb_temp,wind_speed,pitch_angle,active_power,gen_speed,gearbox_oil_temp,transformer_temp,nacelle_temp,hub_temp,yaw_error,vibration_raw,hydraulic_temp,generator_temp,temp_delta_gearbox,temp_trend_24h,rpm_volatility,temp_divergence,power_efficiency,vibration_magnitude,gearbox_oil_temp_delta_6,gearbox_oil_temp_volatility_6,gearbox_oil_temp_delta_24,gearbox_oil_temp_volatility_24,generator_temp_delta_6,generator_temp_volatility_6,generator_temp_delta_24,generator_temp_volatility_24,active_power_delta_6,active_power_volatility_6,active_power_delta_24,active_power_volatility_24,yaw_error_delta_6,yaw_error_volatility_6,yaw_error_delta_24,yaw_error_volatility_24,event_id,event_label,event_start,event_end,state_label,state_name,is_excluded_buffer,target
0,2022-08-04 06:10:00,A,0,22.0,1.7,24.0,-2.560,35.3,41.0,66.333333,30.0,31.0,129.4,22.1,41.0,31.5,19.0,0.0,0.0,-1.0,-0.521067,22.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,0,normal,False,0.0
1,2022-08-04 06:20:00,A,0,22.0,1.7,24.0,-2.556,0.0,41.0,66.333333,30.0,31.0,133.6,0.0,41.0,31.5,19.0,0.0,0.0,-1.0,-0.520252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,0,normal,False,0.0
2,2022-08-04 06:30:00,A,0,22.0,0.9,24.0,-2.712,2.8,41.0,66.333333,30.0,31.0,167.1,5.8,41.0,31.5,19.0,0.0,0.0,-1.0,-3.720165,5.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,0,normal,False,0.0
3,2022-08-04 06:40:00,A,0,22.0,1.5,24.0,-2.548,0.4,41.0,66.000000,29.0,30.0,-49.1,1.9,41.0,31.5,19.0,0.0,0.0,-1.0,-0.754963,1.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,0,normal,False,0.0
4,2022-08-04 06:50:00,A,0,22.0,1.0,24.0,-2.568,0.0,41.0,65.666667,29.0,30.0,-107.3,0.0,41.0,30.5,19.0,0.0,0.0,-1.0,-2.568000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,0,normal,False,0.0



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 46 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   time_stamp                      5000 non-null   datetime64[ns]
 1   farm_id                         5000 non-null   object        
 2   asset_id                        5000 non-null   object        
 3   amb_temp                        5000 non-null   float64       
 4   wind_speed                      5000 non-null   float64       
 5   pitch_angle                     5000 non-null   float64       
 6   active_power                    5000 non-null   float64       
 7   gen_speed                       5000 non-null   float64       
 8   gearbox_oil_temp                5000 non-null   float64       
 9   transformer_temp                5000 non-null   float64       
 10  nacelle_temp                    5000 non-null   float64       
 11  hub

## Basic sample sanity checks

The next cell verifies that key label columns and expected standardized/derived features are present. These checks are not a substitute for full validation, but they help confirm that preprocessing produced the schema required by the modeling pipeline.


In [12]:
# Confirm that preprocessing produced the label fields required for binary experiments.
required_label_columns = ["state_label", "state_name", "target"]
present_label_columns = [c for c in required_label_columns if c in sample_df.columns]

print("Present label columns:", present_label_columns)


# Compare the processed schema against the standardized and derived features from the config.
expected_features = set(processor.standard_features) | set(processor.config.get("derived_features", []))
present_
# Compare the processed schema against the standardized and derived features from the config.
expected_features = sorted(expected_features & set(sample_df.columns))
missing_
# Compare the processed schema against the standardized and derived features from the config.
expected_features = sorted(expected_features - set(sample_df.columns))

schema_check_df = pd.DataFrame(
    [
        {
            "n_sample_rows": len(sample_df),
            "n_columns": sample_df.shape[1],
            "n_present_expected_features": len(present_expected_features),
            "n_missing_expected_features": len(missing_expected_features),
        }
    ]
)
display(schema_check_df)

if missing_expected_features:
    print("Missing expected features:")
    print(missing_expected_features[:50])
else:
    print("All expected mapped/derived features are present in the sample.")


Present label columns: ['state_label', 'state_name', 'target']


NameError: name 'present_' is not defined

## Figure 1: Labeling Strategy Timeline Diagram

This conceptual figure explains how observations are labeled relative to a failure event. The top row shows the broader event timeline, including normal regions, buffer zones, and the pre-failure prediction region. The bottom row shows the three model-specific prediction windows.

The key idea is that each model predicts whether a failure will occur within its future horizon. Longer horizons provide earlier warning, while shorter horizons represent more immediate risk.


In [ ]:
# ------------------------------------------------------------------
# Shared helpers for report figures and tables
# ------------------------------------------------------------------
def prettify_model_name(model_name: str) -> str:
    mapping = {
        "logistic": "Logistic Regression",
        "rf": "Random Forest",
        "random_forest": "Random Forest",
        "xgboost": "XGBoost",
    }
    return mapping.get(str(model_name), str(model_name))


def prettify_experiment_name(experiment_name: str) -> str:
    mapping = {
        "pre_24h": "24h",
        "pre_48h": "48h",
        "pre_72h": "72h",
    }
    return mapping.get(str(experiment_name), str(experiment_name))


def clean_feature_name(feature_name: str) -> str:
    return str(feature_name).replace("_", " ").title()


def save_table_figure(
    df: pd.DataFrame,
    title: str,
    output_path: Path,
    decimals: int = TABLE_DECIMALS,
    index: bool = False,
    figsize: Optional[tuple[float, float]] = None,
) -> None:
    """Render a compact DataFrame as a PNG figure for report insertion."""
    df_to_plot = df.copy()
    numeric_cols = df_to_plot.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        df_to_plot[numeric_cols] = df_to_plot[numeric_cols].round(decimals)

    if not index:
        df_to_plot = df_to_plot.reset_index(drop=True)

    n_rows, n_cols = df_to_plot.shape
    if figsize is None:
        fig_width = max(7.5, 2.05 * n_cols)
        fig_height = max(1.9, 0.42 * (n_rows + 1) + 0.85)
        figsize = (fig_width, fig_height)

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    table = ax.table(
        cellText=df_to_plot.values,
        colLabels=df_to_plot.columns,
        loc="center",
        cellLoc="center",
        bbox=[0.01, 0.10, 0.98, 0.76],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.22)

    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(1.2)
        if row == 0:
            cell.set_text_props(weight="normal")

    ax.set_title(title, fontsize=12, pad=1)
    fig.subplots_adjust(top=0.82, bottom=0.06, left=0.03, right=0.97)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.show()
    plt.close(fig)
    print(f"Saved table figure to: {output_path}")


def save_dataframe_csv(df: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved table data to: {output_path}")


def get_result_row(results_df: pd.DataFrame, model_name: str) -> pd.Series:
    matches = results_df.loc[results_df["model_name"] == model_name]
    if matches.empty:
        raise KeyError(f"Model '{model_name}' was not found in the supplied results.")
    return matches.iloc[0]


report_asset_records: list[dict[str, str]] = []


def has_direct_mapping(mapping_value: Any) -> bool:
    """Return True when a farm has a usable direct mapping for a standardized feature."""
    if mapping_value is None:
        return False
    if isinstance(mapping_value, (list, tuple, set)):
        return any(v not in (None, "") for v in mapping_value)
    return mapping_value != ""


# ------------------------------------------------------------------
# Figure 1: conceptual labeling strategy
# ------------------------------------------------------------------
fig_1_path = REPORT_FIG_DIR / "fig_1_labeling_strategy.png"

pre_buffer_start = -72 - BUFFER_BEFORE_HOURS
event_display_end = FIG_1_EVENT_WINDOW_HOURS
post_buffer_end = event_display_end + BUFFER_AFTER_HOURS

context_segments = [
    (-96, pre_buffer_start, "Normal negative\nregion", "#D9D9D9", True),
    (pre_buffer_start, -72, "", "#CFCFCF", False),
    (-72, 0, "Pre-failure\nprediction region", "#E8E1C4", True),
    (0, event_display_end, "", "#9BBCE0", False),
    (event_display_end, post_buffer_end, "", "#CFCFCF", False),
    (post_buffer_end, 32, "Post-event\nnormal region", "#D9D9D9", True),
]

window_rows = [
    ("72h prediction window", -72, 0, 0.27, "#D95F5F"),
    ("48h prediction window", -48, 0, 0.18, "#F28E2B"),
    ("24h prediction window", -24, 0, 0.09, "#EDC948"),
]

fig, ax = plt.subplots(figsize=(13.5, 5.8))

# Top context timeline
context_y = 0.62
context_height = 0.17
for start, end, label, color, show_label in context_segments:
    ax.add_patch(
        Rectangle(
            (start, context_y),
            end - start,
            context_height,
            facecolor=color,
            edgecolor="black",
            linewidth=1.1,
        )
    )
    if show_label:
        ax.text(
            (start + end) / 2,
            context_y + context_height / 2,
            label,
            ha="center",
            va="center",
            fontsize=10,
        )

# Region annotations above the timeline for narrow segments
annotation_style = dict(arrowstyle="-", linewidth=1.0, color="black")
ax.annotate(
    "Pre-event buffer\n(applied before earliest prediction window)",
    xy=((pre_buffer_start - 72) / 2, context_y + context_height),
    xytext=((pre_buffer_start - 72) / 2, 0.88),
    ha="center",
    va="bottom",
    fontsize=10,
    arrowprops=annotation_style,
)
ax.annotate(
    "Failure event",
    xy=(event_display_end / 2, context_y + context_height),
    xytext=(-1.8, 0.88),
    ha="right",
    va="bottom",
    fontsize=10,
    arrowprops=annotation_style,
)
ax.annotate(
    "Post-event buffer",
    xy=((event_display_end + post_buffer_end) / 2, context_y + context_height),
    xytext=(8.5, 0.88),
    ha="left",
    va="bottom",
    fontsize=10,
    arrowprops=annotation_style,
)

# Bottom model windows
window_height = 0.045
for label, start, end, y, color in window_rows:
    ax.add_patch(
        Rectangle(
            (start, y),
            end - start,
            window_height,
            facecolor=color,
            edgecolor=color,
            linewidth=0,
            alpha=0.98,
        )
    )
    ax.vlines([start, end], y - 0.01, y + window_height + 0.01, colors=color, linewidth=2)
    ax.text(
        start + 2,
        y + window_height / 2,
        label,
        ha="left",
        va="center",
        fontsize=10,
        color="black",
    )

# Failure onset
ax.axvline(0, color="black", linestyle="--", linewidth=1.6)
ax.text(0, 0.40, "Failure onset", ha="center", va="bottom", fontsize=11)

# Explanatory annotations
annotation_box = dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#B0B0B0", alpha=0.95)
ax.text(
    -95,
    0.48,
    "Observations are labeled positive if a failure occurs\nwithin the corresponding prediction window.",
    ha="left",
    va="center",
    fontsize=10,
    bbox=annotation_box,
)
ax.text(
    -95,
    0.03,
    "Prediction windows overlap, so observations closer to failure can be positive\nfor multiple horizons.",
    ha="left",
    va="bottom",
    fontsize=9.5,
)

ax.set_xlim(-100, 33)
ax.set_ylim(0, 1.02)
ax.set_yticks([])
ax.set_xlabel("Hours relative to failure onset")
ax.set_title(
    "Conceptual Labeling Strategy Across 24h, 48h, and 72h Prediction Horizons",
    fontsize=15,
    pad=16,
)
ax.spines[["left", "right", "top"]].set_visible(False)

if SAVE_REPORT_FIGURES:
    fig.savefig(fig_1_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 1", "path": str(fig_1_path)})
    print(f"Saved figure to: {fig_1_path}")

plt.show()
plt.close(fig)

## Load the full processed dataset for consolidated analysis

The full master dataset is loaded after the initial sample checks. Subsequent steps use this table to build horizon-specific modeling datasets, summarize class imbalance, run experiments, and generate report figures.


In [ ]:
master_df = pd.read_parquet(MASTER_DATASET_PATH)
print(f"Master dataset shape: {master_df.shape}")
display(master_df.head(3))


## Figure 2: Cross-Farm Feature Availability and Harmonization Overview

This figure documents a major preprocessing challenge: the wind farms do not share a perfectly consistent sensor schema. Some standardized modeling features map directly to raw farm-specific channels, some are derived from available inputs, and some are unavailable for a farm.

This visualization supports the data-processing section of the report by showing why feature harmonization was necessary before modeling.


In [ ]:
fig_2_path = REPORT_FIG_DIR / "fig_2_harmonization_overview.png"
fig_2_csv_path = REPORT_FIG_DIR / "fig_2_harmonization_overview.csv"

DERIVED_FEATURE_PREREQS = {
    "temp_delta_gearbox": ["gearbox_oil_temp", "amb_temp"],
    "temp_trend_24h": ["gearbox_oil_temp"],
    "rpm_volatility": ["gen_speed"],
    "temp_divergence": ["nacelle_temp", "hub_temp"],
    "power_efficiency": ["wind_speed", "active_power"],
    "vibration_magnitude": ["vibration_raw"],
}

# Keep the report figure focused on the most interpretable features used later in the
# modeling and feature-importance discussion.
HARMONIZATION_FEATURES = [
    "wind_speed",
    "active_power",
    "gen_speed",
    "nacelle_temp",
    "gearbox_oil_temp",
    "generator_temp",
    "hub_temp",
    "hydraulic_temp",
    "vibration_raw",
    "yaw_error",
    "temp_delta_gearbox",
    "power_efficiency",
    "vibration_magnitude",
]


def feature_state_for_farm(feature_name: str, farm_id: str, config: dict[str, Any]) -> str:
    """Classify a standardized feature as Available, Derived, or Unavailable for one farm."""
    farm_sensors = config["farms"][farm_id]["sensors"]

    if feature_name in config.get("standard_features", []):
        return "Available" if has_direct_mapping(farm_sensors.get(feature_name)) else "Unavailable"

    if feature_name in config.get("derived_features", []):
        prereqs = DERIVED_FEATURE_PREREQS.get(feature_name, [])
        if prereqs and all(has_direct_mapping(farm_sensors.get(prereq)) for prereq in prereqs):
            return "Derived"
        return "Unavailable"

    return "Unavailable"


def build_harmonization_overview(
    config: dict[str, Any],
    selected_features: list[str],
) -> pd.DataFrame:
    """Create a feature-by-farm matrix describing harmonization status."""
    records: list[dict[str, str]] = []
    farm_ids = list(config["farms"].keys())

    for feature_name in selected_features:
        feature_label = clean_feature_name(feature_name)
        record = {"feature": feature_label}
        for farm_id in farm_ids:
            record[f"Farm {farm_id}"] = feature_state_for_farm(feature_name, farm_id, config)
        records.append(record)

    return pd.DataFrame(records)


harmonization_df = build_harmonization_overview(
    config=processor.config,
    selected_features=HARMONIZATION_FEATURES,
)

display(harmonization_df)

if SAVE_REPORT_FIGURES:
    save_dataframe_csv(harmonization_df, fig_2_csv_path)
    report_asset_records.append({"asset": "Figure 2 data", "path": str(fig_2_csv_path)})

state_order = ["Unavailable", "Derived", "Available"]
state_to_value = {state: idx for idx, state in enumerate(state_order)}
value_to_code = {"Unavailable": "U", "Derived": "D", "Available": "A"}

plot_matrix = harmonization_df.set_index("feature").replace(state_to_value)
plot_values = plot_matrix.values

fig, ax = plt.subplots(figsize=(8.5, 7.2))
cmap = plt.matplotlib.colors.ListedColormap(["#D9D9D9", "#F2A541", "#2F5D8A"])
ax.imshow(plot_values, aspect="auto", cmap=cmap, vmin=0, vmax=2)

ax.set_xticks(np.arange(plot_matrix.shape[1]))
ax.set_xticklabels(plot_matrix.columns, fontsize=11)
ax.set_yticks(np.arange(plot_matrix.shape[0]))
ax.set_yticklabels(plot_matrix.index, fontsize=11)
ax.set_title("Cross-Farm Feature Availability and Harmonization Overview", fontsize=14, pad=14)
ax.set_xlabel("Wind farm")
ax.set_ylabel("Standardized modeling feature")

# Grid lines for a clean table-like presentation.
ax.set_xticks(np.arange(-0.5, plot_matrix.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, plot_matrix.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.2)
ax.tick_params(which="minor", bottom=False, left=False)

for row_idx in range(plot_values.shape[0]):
    for col_idx in range(plot_values.shape[1]):
        state_name = harmonization_df.iloc[row_idx, col_idx + 1]
        text_color = "white" if state_name == "Available" else "black"
        ax.text(
            col_idx,
            row_idx,
            value_to_code[state_name],
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color=text_color,
        )

legend_handles = [
    Patch(facecolor="#2F5D8A", edgecolor="none", label="Available"),
    Patch(facecolor="#F2A541", edgecolor="none", label="Derived"),
    Patch(facecolor="#D9D9D9", edgecolor="none", label="Unavailable"),
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)

plt.tight_layout()
if SAVE_REPORT_FIGURES:
    plt.savefig(fig_2_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 2", "path": str(fig_2_path)})
    print(f"Saved figure to: {fig_2_path}")
plt.show()
plt.close()

## Preprocessing sanity check: Canonical state distribution

Before converting the canonical preprocessing states into horizon-specific binary targets, the notebook summarizes the state labels present in the full processed dataset. This check verifies that normal rows, pre-failure windows, event rows, and buffer rows are represented as expected.

The report focuses on the simpler binary class imbalance figure below, but this intermediate check is useful for confirming that the event-labeling pipeline behaved correctly.


In [ ]:
state_order = [
    "normal",
    "excluded_buffer",
    "pre_48_72h",
    "pre_24_48h",
    "pre_0_24h",
    "event_occurring",
]

if "state_name" in master_df.columns:
    state_counts = (
        master_df["state_name"]
        .astype(str)
        .value_counts()
        .reindex(state_order, fill_value=0)
        .rename_axis("state_name")
        .reset_index(name="count")
    )
    state_counts["percent"] = state_counts["count"] / state_counts["count"].sum()

    display(state_counts)

    plt.figure(figsize=(9.5, 4.8))
    plt.bar(state_counts["state_name"], state_counts["count"])
    plt.title("Canonical State Distribution in the Full Processed Dataset")
    plt.xlabel("Canonical state")
    plt.ylabel("Row count")
    plt.xticks(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("The processed dataset does not contain 'state_name', so this sanity check was skipped.")

## Modeling helper functions

The following helper functions load previously saved experiment outputs when available and run models only when needed. They also standardize the train/validation/test workflow so each prediction horizon is evaluated consistently.

The notebook uses the project package for the core implementation, while these helpers make the single-notebook submission easier to run top-to-bottom.


In [ ]:
def find_latest_run_dir(experiment_name: str, artifact_root: Path) -> Optional[Path]:
    """Return the newest saved artifact directory for an experiment, if one exists."""
    experiment_dir = artifact_root / experiment_name
    if not experiment_dir.exists():
        return None

    run_dirs = [p for p in experiment_dir.iterdir() if p.is_dir()]
    if not run_dirs:
        return None

    return max(run_dirs, key=lambda p: p.stat().st_mtime)


def load_saved_experiment_bundle(experiment_name: str, artifact_root: Path) -> Optional[dict[str, Any]]:
    """Load saved CSV/JSON artifacts for the newest experiment run if they exist."""
    run_dir = find_latest_run_dir(experiment_name, artifact_root)
    if run_dir is None:
        return None

    results_path = run_dir / "model_comparison_summary.csv"
    threshold_path = run_dir / "threshold_sweep.csv"
    importance_path = run_dir / "feature_importance.csv"
    metadata_path = run_dir / "run_metadata.json"
    summary_metrics_path = run_dir / "summary_metrics.json"

    core_paths = [results_path, threshold_path, importance_path, metadata_path, summary_metrics_path]
    if not all(path.exists() for path in core_paths):
        return None

    results_df = load_dataframe_artifact(results_path)
    threshold_df = load_dataframe_artifact(threshold_path)
    importance_df = load_dataframe_artifact(importance_path)
    run_metadata = load_json_artifact(metadata_path)
    summary_metrics = load_json_artifact(summary_metrics_path)

    best_model_name = summary_metrics.get("best_model_name")
    best_threshold = summary_metrics.get("best_threshold")

    best_row = None
    if "model_name" in results_df.columns and best_model_name in set(results_df["model_name"]):
        best_row = results_df.loc[results_df["model_name"] == best_model_name].iloc[0].to_dict()
    elif not results_df.empty:
        best_row = results_df.iloc[0].to_dict()

    return {
        "experiment_name": experiment_name,
        "source": "saved_artifact",
        "run_dir": run_dir,
        "results_df": results_df,
        "threshold_df": threshold_df,
        "importance_df": importance_df,
        "run_metadata": run_metadata,
        "summary_metrics": summary_metrics,
        "best_model_name": best_model_name,
        "best_threshold": best_threshold,
        "best_row": best_row,
        "train_df": None,
        "val_df": None,
        "test_df": None,
        "feature_summary": run_metadata.get("feature_selection"),
        "detailed_results": None,
    }


def build_experiment_dataset(df: pd.DataFrame, experiment_config: dict[str, Any]) -> pd.DataFrame:
    """Create the experiment-specific binary target and drop excluded-buffer rows."""
    df_exp = WindFarmSplitter.create_binary_target_from_state(
        df=df.copy(),
        positive_states=experiment_config["positive_states"],
    )
    df_exp = df_exp[df_exp["target"].notna()].copy()
    df_exp["target"] = df_exp["target"].astype(int)
    return df_exp


def split_experiment_dataset(
    modeling_df: pd.DataFrame,
    split_method: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Route to the appropriate splitter method."""
    splitter = WindFarmSplitter(random_state=random_state)

    if split_method in {"event_chronological", "event_level_time"}:
        return splitter.get_event_level_time_split(
            modeling_df,
            train_size=train_size,
            val_size=val_size,
            test_size=test_size,
        )
    if split_method in {"group_time", "grouped_time_by_turbine"}:
        return splitter.get_grouped_time_split_by_turbine(
            modeling_df,
            train_size=train_size,
            val_size=val_size,
            test_size=test_size,
        )
    if split_method in {"global_time", "chronological"}:
        return splitter.get_global_time_split(
            modeling_df,
            train_size=train_size,
            val_size=val_size,
            test_size=test_size,
        )

    raise ValueError(f"Unsupported split_method: {split_method}")


def run_experiment_fresh(
    master_df: pd.DataFrame,
    experiment_name: str,
    artifact_root: Path,
    model_names_override: Optional[list[str]] = None,
    numeric_only: bool = True,
    feature_subset: Optional[str] = None,
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
    save_outputs: bool = False,
) -> dict[str, Any]:
    """Run a modeling experiment from scratch using the reusable project modules."""
    experiment = get_experiment_config(experiment_name)
    modeling_df = build_experiment_dataset(master_df, experiment)

    train_df, val_df, test_df = split_experiment_dataset(
        modeling_df=modeling_df,
        split_method=experiment["split_method"],
        train_size=train_size,
        val_size=val_size,
        test_size=test_size,
        random_state=random_state,
    )

    feature_summary = summarize_feature_selection(
        train_df,
        numeric_only=numeric_only,
        feature_subset=feature_subset,
    )
    feature_names = feature_summary["selected_features"]
    validate_no_leakage_columns_in_features(feature_names)

    X_train = build_feature_matrix(train_df, numeric_only=numeric_only, feature_subset=feature_subset)
    y_train = train_df["target"].copy()

    X_val = build_feature_matrix(val_df, numeric_only=numeric_only, feature_subset=feature_subset)
    y_val = val_df["target"].copy()

    X_test = build_feature_matrix(test_df, numeric_only=numeric_only, feature_subset=feature_subset)
    y_test = test_df["target"].copy()

    model_names = model_names_override or list(experiment["models"])
    if not HAS_XGBOOST:
        model_names = [name for name in model_names if name != "xgboost"]

    tournament_rows = []
    detailed_results: dict[str, Any] = {}

    for model_name in model_names:
        model_config = get_model_config(model_name)

        trainer = WindFaultTrainer(
            model_type=model_config["model_type"],
            params=model_config["params"],
            random_state=random_state,
        )

        tuned_threshold = trainer.fit_and_tune(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            optimize_for=experiment["optimize_for"],
        )

        train_metrics = trainer.evaluate_detailed(X_train, y_train)
        val_metrics = trainer.evaluate_detailed(X_val, y_val)
        test_metrics = trainer.evaluate_detailed(X_test, y_test)

        tournament_rows.append(
            {
                "model_name": model_name,
                "model_type": model_config["model_type"],
                "threshold": tuned_threshold,
                "train_precision": train_metrics["precision"],
                "train_recall": train_metrics["recall"],
                "train_f1": train_metrics["f1"],
                "val_precision": val_metrics["precision"],
                "val_recall": val_metrics["recall"],
                "val_f1": val_metrics["f1"],
                "test_precision": test_metrics["precision"],
                "test_recall": test_metrics["recall"],
                "test_f1": test_metrics["f1"],
                "test_roc_auc": test_metrics["roc_auc"],
                "test_pr_auc": test_metrics["pr_auc"],
                "test_balanced_accuracy": test_metrics["balanced_accuracy"],
                "test_specificity": test_metrics["specificity"],
            }
        )

        detailed_results[model_name] = {
            "trainer": trainer,
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "test_metrics": test_metrics,
            "model_config": model_config,
        }

        gc.collect()

    results_df = (
        pd.DataFrame(tournament_rows)
        .sort_values(
            by=["test_f1", "test_recall", "test_precision"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

    if results_df.empty:
        raise RuntimeError(
            f"No model results were produced for experiment '{experiment_name}'. "
            "Check model availability and configuration."
        )

    best_model_name = results_df.iloc[0]["model_name"]
    best_bundle = detailed_results[best_model_name]
    best_trainer = best_bundle["trainer"]
    best_test_metrics = best_bundle["test_metrics"]

    importance_df = best_trainer.get_feature_importance(feature_names).copy()
    threshold_df = build_threshold_sweep_table(
        y_true=y_test,
        y_prob=best_trainer.predict_proba(X_test),
    )

    run_dir = None
    if save_outputs:
        run_dir = ensure_run_output_dir(
            base_output_dir=artifact_root,
            experiment_name=experiment_name,
        )

        metadata = {
            "experiment_name": experiment_name,
            "experiment_config": experiment,
            "data_path": str(MASTER_DATASET_PATH),
            "feature_selection": feature_summary,
            "split_sizes": {
                "train": len(train_df),
                "val": len(val_df),
                "test": len(test_df),
            },
            "best_model_name": best_model_name,
            "best_threshold": float(best_trainer.best_threshold),
            "best_test_metrics": {
                "precision": float(best_test_metrics["precision"]),
                "recall": float(best_test_metrics["recall"]),
                "f1": float(best_test_metrics["f1"]),
                "roc_auc": float(best_test_metrics["roc_auc"]),
                "pr_auc": float(best_test_metrics["pr_auc"]),
                "balanced_accuracy": float(best_test_metrics["balanced_accuracy"]),
                "specificity": float(best_test_metrics["specificity"]),
            },
        }

        summary_metrics = {
            "experiment_name": experiment_name,
            "best_model_name": best_model_name,
            "best_threshold": float(best_trainer.best_threshold),
            "test_metrics": metadata["best_test_metrics"],
        }

        save_dataframe_artifact(results_df, run_dir / "model_comparison_summary.csv")
        save_feature_importance(importance_df, run_dir)
        save_threshold_sweep(threshold_df, run_dir)
        save_run_metadata(metadata, run_dir)
        save_model_metrics(summary_metrics, run_dir)

    return {
        "experiment_name": experiment_name,
        "source": "fresh_run",
        "run_dir": run_dir,
        "results_df": results_df,
        "threshold_df": threshold_df,
        "importance_df": importance_df,
        "run_metadata": {
            "experiment_name": experiment_name,
            "experiment_config": experiment,
            "feature_selection": feature_summary,
            "split_sizes": {
                "train": len(train_df),
                "val": len(val_df),
                "test": len(test_df),
            },
        },
        "summary_metrics": {
            "experiment_name": experiment_name,
            "best_model_name": best_model_name,
            "best_threshold": float(best_trainer.best_threshold),
            "test_metrics": {
                "precision": float(best_test_metrics["precision"]),
                "recall": float(best_test_metrics["recall"]),
                "f1": float(best_test_metrics["f1"]),
                "roc_auc": float(best_test_metrics["roc_auc"]),
                "pr_auc": float(best_test_metrics["pr_auc"]),
                "balanced_accuracy": float(best_test_metrics["balanced_accuracy"]),
                "specificity": float(best_test_metrics["specificity"]),
            },
        },
        "best_model_name": best_model_name,
        "best_threshold": float(best_trainer.best_threshold),
        "best_row": results_df.iloc[0].to_dict(),
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "feature_summary": feature_summary,
        "detailed_results": detailed_results,
    }


def get_experiment_bundle(
    master_df: pd.DataFrame,
    experiment_name: str,
    artifact_root: Path,
    reuse_saved_model_artifacts: bool = True,
    save_fresh_modeling_artifacts: bool = False,
) -> dict[str, Any]:
    """Load a saved experiment bundle if available, otherwise run it fresh."""
    if reuse_saved_model_artifacts:
        saved_bundle = load_saved_experiment_bundle(experiment_name, artifact_root)
        if saved_bundle is not None:
            return saved_bundle

    return run_experiment_fresh(
        master_df=master_df,
        experiment_name=experiment_name,
        artifact_root=artifact_root,
        model_names_override=MODEL_NAMES_OVERRIDE,
        numeric_only=NUMERIC_ONLY,
        feature_subset=FEATURE_SUBSET,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        save_outputs=SAVE_FRESH_MODELING_ARTIFACTS,
    )


## Run or reuse the three horizon experiments

This section evaluates the same modeling pipeline across 24h, 48h, and 72h prediction windows. Each experiment builds a horizon-specific binary target from the canonical state labels, trains the selected models, tunes classification thresholds on validation data, and reports test-set performance.

Saved artifacts are reused when possible. This prevents the notebook from repeatedly retraining models during editing while still allowing a full rebuild when needed.


In [ ]:
experiment_bundles: dict[str, dict[str, Any]] = {}

for experiment_name in EXPERIMENT_NAMES:
    print(f"\n=== {experiment_name} ===")
    bundle = get_experiment_bundle(
        master_df=master_df,
        experiment_name=experiment_name,
        artifact_root=ARTIFACT_ROOT,
        reuse_saved_model_artifacts=REUSE_SAVED_MODEL_ARTIFACTS,
        save_fresh_modeling_artifacts=SAVE_FRESH_MODELING_ARTIFACTS,
    )
    experiment_bundles[experiment_name] = bundle

    print(f"Source: {bundle['source']}")
    if bundle.get("run_dir") is not None:
        print(f"Run directory: {bundle['run_dir']}")
    display(bundle["results_df"])


## Figure 3: Class Imbalance Distribution

Failure prediction is a rare-event classification problem. The next figure shows the binary class distribution for the representative modeling dataset used in the report. Bar heights show class proportions, while annotations show raw row counts.

This figure explains why accuracy is not a meaningful evaluation metric and why precision, recall, F1, and threshold tuning are emphasized instead.


In [ ]:
report_experiment_config = get_experiment_config(REPORT_EXPERIMENT_NAME)

# Build the representative binary modeling dataset from the canonical state labels.
# This mirrors the label policy used by the 72h report experiment.
report_modeling_df = build_experiment_dataset(master_df, report_experiment_config).copy()

class_balance_df = (
    report_modeling_df["target"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)
class_balance_df["class_name"] = class_balance_df["target"].map({0: "Non-failure", 1: "Failure"})
class_balance_df["percent"] = class_balance_df["count"] / class_balance_df["count"].sum()

display(class_balance_df[["class_name", "count", "percent"]])

fig_3_path = REPORT_FIG_DIR / "fig_3_class_imbalance_distribution.png"

fig, ax = plt.subplots(figsize=(7.5, 5))
bars = ax.bar(class_balance_df["class_name"], class_balance_df["percent"])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"Class Distribution for {prettify_experiment_name(REPORT_EXPERIMENT_NAME)} Modeling Dataset")
ax.set_xlabel("Class")
ax.set_ylabel("Proportion of modeling rows")

# Keep the y-axis normalized while leaving room for raw-count annotations.
y_max = class_balance_df["percent"].max()
ax.set_ylim(0, min(1.08, y_max + 0.12))

# Bar labels show counts rather than percentages so the visual emphasizes imbalance
# while still reporting the actual number of rows in each class.
y_offset = max(class_balance_df["percent"].max() * 0.015, 0.01)
for bar, count in zip(bars, class_balance_df["count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_offset,
        f"n = {int(count):,}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

if SAVE_REPORT_FIGURES:
    fig.savefig(fig_3_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 3", "path": str(fig_3_path)})
    print(f"Saved figure to: {fig_3_path}")

plt.show()
plt.close(fig)


## Figure 4: Model Performance Comparison Table

This table compares Logistic Regression, Random Forest, and XGBoost on the representative 72h prediction horizon. The purpose is to show how model choice affects the precision-recall tradeoff on the same prediction task.

The values reported here are test-set metrics after validation-based threshold selection.


In [ ]:
# Use the representative 72h experiment to compare model families on the same target.
results_72h_df = experiment_bundles[REPORT_EXPERIMENT_NAME]["results_df"].copy()

model_table_df = (
    results_72h_df.loc[:, ["model_name", "test_precision", "test_recall", "test_f1"]]
    .rename(
        columns={
            "model_name": "Model",
            "test_precision": "Precision",
            "test_recall": "Recall",
            "test_f1": "F1",
        }
    )
)
model_table_df["Model"] = model_table_df["Model"].map(prettify_model_name)
model_table_df = model_table_df.sort_values("F1", ascending=False).reset_index(drop=True)

display(model_table_df)

fig_4_path = REPORT_FIG_DIR / "fig_4_model_performance_comparison_table.png"
fig_4_csv_path = REPORT_FIG_DIR / "fig_4_model_performance_comparison_table.csv"

if SAVE_REPORT_FIGURES:
    save_table_figure(
        model_table_df,
        title=f"Model Performance Comparison ({prettify_experiment_name(REPORT_EXPERIMENT_NAME)} Horizon)",
        output_path=fig_4_path,
        index=False,
        figsize=(11.5, 2.6),
    )
    save_dataframe_csv(model_table_df, fig_4_csv_path)
    report_asset_records.append({"asset": "Figure 4", "path": str(fig_4_path)})
    report_asset_records.append({"asset": "Figure 4 data", "path": str(fig_4_csv_path)})

## Figure 5: Prediction Window Comparison Table

This table isolates the effect of prediction horizon by comparing XGBoost performance across 24h, 48h, and 72h windows. Similar performance across horizons supports the interpretation that failure signals are diffuse over time rather than concentrated immediately before failure onset.


In [ ]:
# Hold the model fixed and vary only the prediction horizon.
xgb_rows = []

for experiment_name, bundle in experiment_bundles.items():
    results_df = bundle["results_df"].copy()
    if REPORT_MODEL_NAME not in set(results_df["model_name"]):
        continue

    row = get_result_row(results_df, REPORT_MODEL_NAME)
    xgb_rows.append(
        {
            "Window": prettify_experiment_name(experiment_name),
            "Precision": row["test_precision"],
            "Recall": row["test_recall"],
            "F1": row["test_f1"],
        }
    )

xgb_window_table_df = pd.DataFrame(xgb_rows).sort_values("Window").reset_index(drop=True)
display(xgb_window_table_df)

fig_5_path = REPORT_FIG_DIR / "fig_5_prediction_window_comparison_table.png"
fig_5_csv_path = REPORT_FIG_DIR / "fig_5_prediction_window_comparison_table.csv"

if SAVE_REPORT_FIGURES:
    save_table_figure(
        xgb_window_table_df,
        title="XGBoost Performance Across Prediction Windows",
        output_path=fig_5_path,
        index=False,
        figsize=(11.5, 2.6),
    )
    save_dataframe_csv(xgb_window_table_df, fig_5_csv_path)
    report_asset_records.append({"asset": "Figure 5", "path": str(fig_5_path)})
    report_asset_records.append({"asset": "Figure 5 data", "path": str(fig_5_csv_path)})

## Prepare report diagnostics for detailed visualizations

The saved experiment summaries are sufficient for aggregate performance tables, but the threshold sweep, temporal prediction timeline, and feature importance plot require row-level predictions and model-specific diagnostics. This section loads cached diagnostics if available or builds them with a lightweight fresh run for the selected report model and horizon.


In [ ]:
def get_report_diagnostics(
    master_df: pd.DataFrame,
    experiment_name: str = REPORT_EXPERIMENT_NAME,
    model_name: str = REPORT_MODEL_NAME,
    force_rebuild: bool = FORCE_REBUILD_REPORT_DIAGNOSTICS,
) -> dict[str, Any]:
    """Load cached report diagnostics or build them with one fresh model run."""
    diag_dir = REPORT_DIAGNOSTICS_DIR / experiment_name / model_name
    diag_dir.mkdir(parents=True, exist_ok=True)

    predictions_path = diag_dir / "test_predictions.parquet"
    validation_threshold_path = diag_dir / "validation_threshold_sweep.csv"
    feature_importance_path = diag_dir / "feature_importance.csv"
    metadata_path = diag_dir / "diagnostic_metadata.json"

    if (
        not force_rebuild
        and predictions_path.exists()
        and validation_threshold_path.exists()
        and feature_importance_path.exists()
        and metadata_path.exists()
    ):
        return {
            "source": "cached_report_diagnostics",
            "test_predictions_df": pd.read_parquet(predictions_path),
            "validation_threshold_df": pd.read_csv(validation_threshold_path),
            "feature_importance_df": pd.read_csv(feature_importance_path),
            "metadata": json.loads(metadata_path.read_text()),
        }

    fresh_bundle = run_experiment_fresh(
        master_df=master_df,
        experiment_name=experiment_name,
        artifact_root=ARTIFACT_ROOT,
        model_names_override=[model_name],
        numeric_only=NUMERIC_ONLY,
        feature_subset=FEATURE_SUBSET,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        save_outputs=False,
    )

    trainer = fresh_bundle["detailed_results"][model_name]["trainer"]
    val_df = fresh_bundle["val_df"].copy()
    test_df = fresh_bundle["test_df"].copy()

    X_val = build_feature_matrix(val_df, numeric_only=NUMERIC_ONLY, feature_subset=FEATURE_SUBSET)
    y_val = val_df["target"].copy()

    X_test = build_feature_matrix(test_df, numeric_only=NUMERIC_ONLY, feature_subset=FEATURE_SUBSET)

    validation_threshold_df = build_threshold_sweep_table(
        y_true=y_val,
        y_prob=trainer.predict_proba(X_val),
    )

    test_predictions_df = test_df.copy()
    test_predictions_df["predicted_probability"] = trainer.predict_proba(X_test)
    test_predictions_df["predicted_label"] = (
        test_predictions_df["predicted_probability"] >= float(trainer.best_threshold)
    ).astype(int)

    feature_importance_df = trainer.get_feature_importance(
        fresh_bundle["feature_summary"]["selected_features"]
    ).copy()

    metadata = {
        "experiment_name": experiment_name,
        "model_name": model_name,
        "best_threshold": float(trainer.best_threshold),
        "source": "fresh_report_run",
        "n_val_rows": int(len(val_df)),
        "n_test_rows": int(len(test_df)),
    }

    test_predictions_df.to_parquet(predictions_path, index=False)
    validation_threshold_df.to_csv(validation_threshold_path, index=False)
    feature_importance_df.to_csv(feature_importance_path, index=False)
    metadata_path.write_text(json.dumps(metadata, indent=2))

    return {
        "source": "fresh_report_run",
        "test_predictions_df": test_predictions_df,
        "validation_threshold_df": validation_threshold_df,
        "feature_importance_df": feature_importance_df,
        "metadata": metadata,
    }


report_diag = get_report_diagnostics(master_df=master_df)
print("Report diagnostic source:", report_diag["source"])
display(pd.DataFrame([report_diag["metadata"]]))

## Figure 6: Threshold Sweep Curve

This figure shows validation-set precision, recall, and F1 across classification thresholds. It demonstrates why the default threshold of 0.5 is not appropriate for this imbalanced problem and why threshold tuning is part of the final evaluation pipeline.


In [ ]:
validation_threshold_df = report_diag["validation_threshold_df"].copy()
best_threshold = report_diag["metadata"]["best_threshold"]

display(
    validation_threshold_df.sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False],
    ).head(15)
)

fig_6_path = REPORT_FIG_DIR / "fig_6_threshold_sweep_curve.png"

plt.figure(figsize=(10, 6))
plt.plot(validation_threshold_df["threshold"], validation_threshold_df["precision"], label="Precision", linewidth=2)
plt.plot(validation_threshold_df["threshold"], validation_threshold_df["recall"], label="Recall", linewidth=2)
plt.plot(validation_threshold_df["threshold"], validation_threshold_df["f1"], label="F1", linewidth=2)
plt.axvline(best_threshold, linestyle="--", linewidth=1.5, label=f"Tuned threshold = {best_threshold:.2f}")
plt.title(f"Validation Threshold Sweep | {prettify_model_name(REPORT_MODEL_NAME)} | {prettify_experiment_name(REPORT_EXPERIMENT_NAME)}")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend(frameon=False)
plt.tight_layout()
if SAVE_REPORT_FIGURES:
    plt.savefig(fig_6_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 6", "path": str(fig_6_path)})
    print(f"Saved figure to: {fig_6_path}")
plt.show()
plt.close()

## Figure 7: Temporal Prediction Timeline

This figure plots predicted failure probability over time for a representative event window. It is intended to show whether the model produces temporally structured warnings rather than isolated random spikes.

Elevated probabilities before the failure event support the project’s central interpretation: turbine failures are better understood as gradual degradation processes than as instantaneous events.


In [ ]:
timeline_predictions_df = report_diag["test_predictions_df"].copy()

for dt_col in ["time_stamp", "event_start", "event_end"]:
    if dt_col in timeline_predictions_df.columns:
        timeline_predictions_df[dt_col] = pd.to_datetime(timeline_predictions_df[dt_col], errors="coerce")

group_cols = [c for c in ["farm_id", "asset_id", "event_id", "event_label", "event_start"] if c in timeline_predictions_df.columns]

if not group_cols:
    raise KeyError(
        "The prediction timeline requires event metadata columns such as "
        "'farm_id', 'asset_id', 'event_id', and 'event_start'."
    )

event_summary_df = (
    timeline_predictions_df.dropna(subset=["event_start"])
    .groupby(group_cols, dropna=False)
    .agg(
        n_rows=("predicted_probability", "size"),
        n_positive=("target", "sum"),
        max_probability=("predicted_probability", "max"),
        mean_probability=("predicted_probability", "mean"),
    )
    .reset_index()
)

positive_event_summary_df = event_summary_df.loc[event_summary_df["n_positive"] > 0].copy()
if positive_event_summary_df.empty:
    positive_event_summary_df = event_summary_df.copy()

# Choose a representative event with actual positive-labeled rows, then prefer
# stronger model response within that subset. This is more defensible than
# selecting solely on the single highest peak probability.
positive_event_summary_df = positive_event_summary_df.sort_values(
    by=["n_positive", "mean_probability", "max_probability", "event_start"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(positive_event_summary_df.head(10))

selected_event = positive_event_summary_df.iloc[0].to_dict()

mask = pd.Series(True, index=timeline_predictions_df.index)
for col in group_cols:
    value = selected_event[col]
    if pd.isna(value):
        mask &= timeline_predictions_df[col].isna()
    else:
        mask &= timeline_predictions_df[col] == value

selected_timeline_df = timeline_predictions_df.loc[mask].copy().sort_values("time_stamp")

selected_event_start = pd.to_datetime(selected_event["event_start"])
selected_timeline_df["hours_from_event_start"] = (
    (selected_timeline_df["time_stamp"] - selected_event_start).dt.total_seconds() / 3600.0
)

selected_timeline_df = selected_timeline_df[
    (selected_timeline_df["hours_from_event_start"] >= -TIMELINE_HOURS_BEFORE)
    & (selected_timeline_df["hours_from_event_start"] <= TIMELINE_HOURS_AFTER)
].copy()

fig_7_path = REPORT_FIG_DIR / "fig_7_temporal_prediction_timeline.png"

plt.figure(figsize=(10.5, 5.8))
plt.plot(
    selected_timeline_df["hours_from_event_start"],
    selected_timeline_df["predicted_probability"],
    linewidth=2,
    label="Predicted probability",
)
plt.axvline(0, linestyle="--", linewidth=1.5, label="Failure onset")
plt.axhline(best_threshold, linestyle=":", linewidth=1.5, label=f"Tuned threshold = {best_threshold:.2f}")

positive_rows_df = selected_timeline_df.loc[selected_timeline_df["target"] == 1]
if not positive_rows_df.empty:
    plt.scatter(
        positive_rows_df["hours_from_event_start"],
        positive_rows_df["predicted_probability"],
        s=28,
        label="Positive-labeled rows",
    )

event_label_text = selected_event.get("event_label", "unknown")
farm_text = selected_event.get("farm_id", "unknown")
asset_text = selected_event.get("asset_id", "unknown")
event_id_text = selected_event.get("event_id", "unknown")

plt.title(
    f"Temporal Prediction Behavior | Event {event_id_text} | Farm {farm_text} | Turbine {asset_text}"
)
plt.xlabel("Hours relative to failure onset")
plt.ylabel("Predicted probability")
plt.legend(frameon=False)
plt.tight_layout()

if SAVE_REPORT_FIGURES:
    plt.savefig(fig_7_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 7", "path": str(fig_7_path)})
    print(f"Saved figure to: {fig_7_path}")

plt.show()
plt.close()

display(
    selected_timeline_df.loc[
        :,
        ["time_stamp", "hours_from_event_start", "predicted_probability", "target", "state_name"]
        if "state_name" in selected_timeline_df.columns
        else ["time_stamp", "hours_from_event_start", "predicted_probability", "target"]
    ].head(25)
)

## Figure 8: Feature Importance Plot

Feature importance from the XGBoost model helps evaluate whether the model is relying on physically meaningful signals. Important features related to vibration, temperature, wind speed, power efficiency, and mechanical behavior are consistent with the predictive-maintenance framing of the project.


In [ ]:
importance_df = report_diag["feature_importance_df"].copy()

value_col = "importance" if "importance" in importance_df.columns else "coefficient"
plot_importance_df = (
    importance_df
    .sort_values(value_col, ascending=False)
    .head(TOP_N_FEATURES)
    .iloc[::-1]
    .copy()
)
plot_importance_df["feature_display"] = plot_importance_df["feature"].map(clean_feature_name)

display(plot_importance_df.loc[:, ["feature", value_col]].sort_values(value_col, ascending=False))

fig_8_path = REPORT_FIG_DIR / "fig_8_feature_importance_plot.png"

plt.figure(figsize=(10, max(6, 0.35 * len(plot_importance_df))))
plt.barh(plot_importance_df["feature_display"], plot_importance_df[value_col])
plt.title(
    f"Top {TOP_N_FEATURES} Features | {prettify_model_name(REPORT_MODEL_NAME)} | "
    f"{prettify_experiment_name(REPORT_EXPERIMENT_NAME)}"
)
plt.xlabel("Importance" if value_col == "importance" else "Coefficient")
plt.ylabel("Feature")
plt.tight_layout()

if SAVE_REPORT_FIGURES:
    plt.savefig(fig_8_path, dpi=300, bbox_inches="tight")
    report_asset_records.append({"asset": "Figure 8", "path": str(fig_8_path)})
    print(f"Saved figure to: {fig_8_path}")

plt.show()
plt.close()

## Report Outputs

The following manifest lists the figures and table exports generated by this notebook for use in the final report. Figures are saved with stable filenames so they can be inserted into the report without manually tracking notebook output cells.


In [ ]:
report_asset_manifest_df = (
    pd.DataFrame(report_asset_records)
    .drop_duplicates()
    .sort_values("asset")
    .reset_index(drop=True)
)
display(report_asset_manifest_df)

## Summary

This notebook demonstrated the full wind turbine fault detection pipeline from raw-data preprocessing through model evaluation and report-output generation.

Key takeaways:

- Cross-farm schema inconsistency required feature harmonization before modeling.
- Event-based labeling converted the time-series problem into horizon-specific binary classification tasks.
- Failure observations were rare, making precision, recall, F1, and threshold tuning more appropriate than accuracy.
- XGBoost provided the strongest overall performance among the evaluated models.
- Similar performance across 24h, 48h, and 72h windows suggests that predictive signals are distributed over time.
- Feature importance and temporal prediction behavior support the interpretation that the model is detecting gradual degradation patterns rather than isolated anomalies.

The notebook is designed to complement the final report by making the implementation reproducible and by saving the core visuals used to support the report narrative.


## Cleanup

The final cell releases large intermediate DataFrames from memory. This is not required for correctness, but it is helpful when working with large processed datasets in a notebook environment.


In [ ]:
for var_name in [
    "sample_df",
    "master_df",
    "report_modeling_df",
    "timeline_predictions_df",
    "selected_timeline_df",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
print("Cleanup complete.")